# Single-channel nanodomain identification

Runs one SMLM localization table (one color / one histone mark) through the density-independent DBSCAN pipeline:

1. **Preprocess**: axial filter (only if the data has z), nucleus mask, radial contours
2. **Inverse Monte Carlo calibration**: simulate a nucleus matching this cell (radial profile, within-domain RDF/ADF, inter-domain g(r)) with known ground-truth domains, subsample it to a range of densities, and fit `eps(density)` and `min_samples(density)` so DBSCAN recovers the true domain size at every density
3. **Density-corrected DBSCAN**: evaluate those fits at this cell's density and cluster
4. **Nanodomain characteristics**: size, localization count, centroid, hull area, per domain

It's the same chain as `drivers/01_preprocess.py` → `02_calibrate.py` → `03_cluster.py`, but for a single channel and a single cell. Works with 2D data (no `z [nm]` column, or a constant one): the axial filter is skipped and everything is clustered in XY. Step 2 is slow; set `QUICK = True` for a smoke test, or point `CALIBRATION_JSON` at a saved calibration to skip it.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

REPO_ROOT = Path.cwd()  # run from the repo root (c:/hmt_v2)
sys.path.insert(0, str(REPO_ROOT))
import hmt_v3 as hmt

## Settings

In [ ]:
# --- Input -------------------------------------------------------------------
DATA_PATH = REPO_ROOT / "test_data" / "hela_1h_001_0.csv"  # columns x [nm], y [nm] (z [nm] optional)
LABEL = "EdU"
COLOR = "#3335BD"
OUTPUT_DIR = REPO_ROOT / "analysis_output_single" / DATA_PATH.stem

# Reuse a calibration saved by an earlier run of this notebook (or drivers/02_calibrate.py)
# instead of recalibrating. None = calibrate on this cell.
CALIBRATION_JSON = None

QUICK = False  # smaller crop / fewer fractions / coarser min_samples grid, for a smoke test

# --- Preprocess --------------------------------------------------------------
AXIAL_PADDING_NM = 1
MASK_BIN_NM = 50
MASK_SIGMA = 4.0
MASK_THRESH = "auto"     # or a number (smoothed locs per pixel)
MASK_AUTO_FACTOR = 0.5   # "auto" = this x Otsu threshold of the smoothed density map
N_CONTOUR_BANDS = 100    # radial contour bands: 0 = core ... N_CONTOUR_BANDS - 1 = periphery

# --- Calibration ---------------------------------------------------------------
USE_Z = True                 # cluster in 3D; forced False if the data has no z. Must match the calibration's
DOMAIN_SCALE_NM = 200.0      # within-domain cutoff radius; None = read off g(r) (tends to underestimate)
NOISE_FRACTION = None        # None = estimate from g(r)
SEED_PLACEMENT = "thomas"    # "thomas" | "inhibition" | "poisson"
CROP_SIZE_NM = 5000.0
CROP_MIN_INSIDE = 0.95
DENSITY_FRACTIONS = np.round(np.linspace(1.0, 0.1, 10), 3)
MIN_SAMPLES_RANGE = range(5, 75, 5)
SIZE_COST_THRESHOLD_NM = 50.0
SIZE_PERCENTILE = 75.0
RANDOM_SEED = 0

# --- Clustering ----------------------------------------------------------------
MIN_EPS_NM = 10.0
MIN_MIN_SAMPLES = 3
EXTRAPOLATION_TOLERANCE = 0.05
MIN_DOMAIN_LOCS = 10  # domains smaller than this are left out of the size summaries

if QUICK:
    CROP_SIZE_NM = 3000.0
    DENSITY_FRACTIONS = np.round(np.linspace(1.0, 0.2, 5), 3)
    MIN_SAMPLES_RANGE = range(5, 45, 10)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Preprocess: axial filter, nucleus mask, radial contours

If the table has no `z [nm]` column (or z is constant), the axial filter is skipped, a constant z = 0 placeholder is added because several `hmt_v3` functions read that column, and `USE_Z` is set to False so z never enters the clustering.

`binarize_nucleus` / `auto_mask_threshold` take two channels (they concatenate them to build the mask). With one channel, the second argument is an empty table.

Check the mask covers the whole nucleus. If it doesn't, set `MASK_THRESH` to a number.

In [ ]:
raw = pd.read_csv(DATA_PATH)
HAS_Z = "z [nm]" in raw.columns and raw["z [nm]"].nunique() > 1
if HAS_Z:
    locs = hmt.preprocess.filter_axial(raw, padding=AXIAL_PADDING_NM)
else:
    locs = raw.assign(**{"z [nm]": 0.0})
    USE_Z = False
    print("no z information: axial filter skipped, clustering in 2D (USE_Z = False)")
empty = locs.iloc[:0]

thresh = MASK_THRESH
if thresh == "auto":
    thresh = hmt.preprocess.auto_mask_threshold(locs, empty, bin_size=MASK_BIN_NM, sigma=MASK_SIGMA,
                                                factor=MASK_AUTO_FACTOR)

binary_mask, masked, _ = hmt.preprocess.binarize_nucleus(
    locs, empty, thresh=thresh, bin_size=MASK_BIN_NM, sigma=MASK_SIGMA, num_nuclei=1, show_plots=True)
# Mask grid origin: min x/y of the PRE-mask table (every later step that maps locs onto the mask needs it)
x_min, y_min = hmt.preprocess.mask_origin(locs)

_, contour_bands = hmt.preprocess.create_radial_contours(binary_mask, num_bands=N_CONTOUR_BANDS, show_plots=True)

area_nm2 = float(binary_mask.sum()) * MASK_BIN_NM ** 2
print(f"mask threshold {thresh:.3f} | mask area {area_nm2 / 1e6:.0f} um^2")
print(f"{len(masked):,} / {len(raw):,} localizations kept "
      f"({len(locs):,} before masking) | density {len(masked) / area_nm2 * 1e6:.0f} locs/um^2")

## 2. Inverse Monte Carlo calibration

Skipped if `CALIBRATION_JSON` is set.

1. `generate_nucleus` simulates a nucleus matching this cell, with known per-localization domain labels. Check the pair-correlation plot: simulated g(r) and L(r) should overlay the real ones.
2. A `CROP_SIZE_NM` square is cropped where the simulated density matches the nucleus mean, since DBSCAN over the whole nucleus for every parameter pair is too slow. Crop density = locs / in-mask area, measured the same way as the real cell's density.
3. The crop is subsampled to `DENSITY_FRACTIONS`. `optimize_dbscan_size_matching` finds the (eps, min_samples) that best recovers the ground-truth domain size at each density, then fits `eps = a * density^b` and `min_samples = m * density + c`.

In [ ]:
def choose_crop(sim_df, binary_mask, x_min, y_min, bin_nm, size_nm, min_inside):
    # Square window, >= min_inside inside the mask, whose density (per in-mask area) is closest to the
    # simulated nucleus mean. Shrinks the window if the nucleus can't hold it. Same as drivers/02_calibrate.py.
    mask = binary_mask.astype(float)
    nx, ny = mask.shape
    x_edges = x_min + np.arange(nx + 1) * bin_nm
    y_edges = y_min + np.arange(ny + 1) * bin_nm
    counts, _, _ = np.histogram2d(sim_df["x [nm]"], sim_df["y [nm]"], bins=(x_edges, y_edges))
    target = len(sim_df) / (mask.sum() * bin_nm ** 2)

    def box_sums(img, w):
        s = np.pad(img, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
        return s[w:, w:] - s[:-w, w:] - s[w:, :-w] + s[:-w, :-w]

    size = size_nm
    while size >= 1500:
        w = int(round(size / bin_nm))
        if w < nx and w < ny:
            inside = box_sums(mask, w)
            n = box_sums(counts, w)
            ok = inside >= min_inside * w * w
            if ok.any():
                density = np.where(ok, n / np.maximum(inside * bin_nm ** 2, 1), np.nan)
                i, j = np.unravel_index(np.nanargmin(np.abs(density - target)), density.shape)
                bounds = (x_edges[i], x_edges[i + w], y_edges[j], y_edges[j + w])
                return bounds, float(inside[i, j] * bin_nm ** 2)
        size *= 0.8
    raise ValueError("nucleus too small for any calibration crop >= 1.5 um")


def crop(df, bounds):
    xmin, xmax, ymin, ymax = bounds
    return df[df["x [nm]"].between(xmin, xmax, inclusive="left") &
              df["y [nm]"].between(ymin, ymax, inclusive="left")].copy()


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, range, np.ndarray)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    return obj

In [ ]:
if CALIBRATION_JSON is not None:
    with open(CALIBRATION_JSON) as f:
        calibration = json.load(f)
    print(f"loaded calibration from {CALIBRATION_JSON}")
    if calibration.get("use_z", USE_Z) != USE_Z:
        raise ValueError(f"calibration was fit with use_z={calibration['use_z']}, but USE_Z={USE_Z}")
    fraction_dfs = None
else:
    _, sim_df, sim_info = hmt.simulate.generate_nucleus(
        masked, contour_bands, x_min, y_min, px_size=MASK_BIN_NM,
        domain_scale=DOMAIN_SCALE_NM, noise_fraction=NOISE_FRACTION, seed_placement=SEED_PLACEMENT,
        rng=np.random.default_rng(RANDOM_SEED), verbose=True)
    hmt.visualize.plot_pair_correlation(sim_info["pcf_real"], sim_info["pcf_noisy"], r_domain=sim_info["r_domain_nm"])

In [ ]:
if CALIBRATION_JSON is None:
    crop_bounds, crop_area_nm2 = choose_crop(sim_df, binary_mask, x_min, y_min, MASK_BIN_NM,
                                             CROP_SIZE_NM, CROP_MIN_INSIDE)
    crop_df = crop(sim_df, crop_bounds)
    print(f"crop {crop_bounds[1] - crop_bounds[0]:.0f} nm square: {len(crop_df):,} locs, "
          f"{len(crop_df) / crop_area_nm2 * 1e6:.0f} locs/um^2 (nucleus mean {len(sim_df) / area_nm2 * 1e6:.0f})")

    packing = hmt.calibrate.diagnose_domain_packing(crop_df, domain_scale=DOMAIN_SCALE_NM, use_z=USE_Z)
    print(f"{packing['n_domains']} ground-truth domains in crop, "
          f"{packing['frac_overlapping']:.0%} overlap their nearest neighbour")

    rng = np.random.default_rng(RANDOM_SEED)
    fraction_dfs = [crop_df if f >= 1.0 else crop_df.sample(frac=f, random_state=int(rng.integers(2**31)))
                    for f in DENSITY_FRACTIONS]

    fig, ax = plt.subplots(figsize=(6, 6))
    hmt.visualize.plot_nanodomain_2d(crop_df, title="Simulated crop (ground truth, noise in gray)",
                                     point_size=1, plot_seeds=False, ax=ax)
    plt.show()

In [ ]:
if CALIBRATION_JSON is None:
    calibration = hmt.calibrate.optimize_dbscan_size_matching(
        fraction_dfs, crop_area_nm2, min_samples_range=MIN_SAMPLES_RANGE, percentile=SIZE_PERCENTILE,
        cost_threshold=SIZE_COST_THRESHOLD_NM, use_z=USE_Z, show_plot=True, verbose=False)
    calibration.update({
        "source": str(DATA_PATH),
        "calibration_area_nm2": crop_area_nm2,
        "crop_bounds_nm": crop_bounds,
        "use_z": USE_Z,
        "density_fractions": DENSITY_FRACTIONS,
        "min_samples_range": list(MIN_SAMPLES_RANGE),
        "domain_scale_nm": DOMAIN_SCALE_NM,
        "generate_nucleus": {k: v for k, v in sim_info.items() if np.isscalar(v) or v is None},
        "gt_frac_domains_overlapping": packing["frac_overlapping"],
        "converged": len(calibration["densities"]) >= 2,
    })
    cal_path = OUTPUT_DIR / "calibration.json"
    with open(cal_path, "w") as f:
        json.dump(to_jsonable(calibration), f, indent=2)
    print(f"calibration -> {cal_path}")

n_fit = len(calibration["densities"])
print(f"eps = {calibration['eps_coef']:.3g} * density^{calibration['eps_exp']:.3f}  (R^2 {calibration['eps_r_squared']:.3f})")
print(f"min_samples = {calibration['min_samples_slope']:.4g} * density + {calibration['min_samples_intercept']:.2f}")
print(f"{n_fit} fractions under the {SIZE_COST_THRESHOLD_NM:.0f} nm size-error cutoff | "
      f"ground-truth p{SIZE_PERCENTILE:.0f} diameter {calibration['gt_target']:.0f} nm")
if n_fit < 2:
    print("WARNING: calibration did not converge -- try a different DOMAIN_SCALE_NM or a larger SIZE_COST_THRESHOLD_NM")

### Check: is measured size independent of density?

Clusters every density fraction two ways: with **fixed** parameters (the corrected ones at full density) and with **density-corrected** parameters. Corrected sizes should stay flat across density (R² near 0). Fixed sizes should drift. Only runs when this notebook did the calibration.

In [ ]:
def mean_sizes(clustered_df):
    nd = hmt.cluster.calculate_nanodomain_characteristics(clustered_df, percentile=SIZE_PERCENTILE)
    nd = nd[nd["n_locs"] >= MIN_DOMAIN_LOCS]
    if nd.empty:
        return np.nan, np.nan, 0
    return nd["bb_size_2D"].mean(), nd["geometric_diameter_2D"].mean(), len(nd)


if fraction_dfs is not None and n_fit >= 2:
    kw = dict(use_z=USE_Z, min_eps=MIN_EPS_NM, min_min_samples=MIN_MIN_SAMPLES)
    _, fixed = hmt.cluster.cluster_dbscan_density_corrected(fraction_dfs[0], calibration, crop_area_nm2, **kw)
    rows = []
    for frac, df in zip(DENSITY_FRACTIONS, fraction_dfs):
        fixed_df = hmt.cluster.cluster_dbscan(df, eps=fixed["eps"], min_samples=fixed["min_samples"], use_z=USE_Z)
        corr_df, corr = hmt.cluster.cluster_dbscan_density_corrected(df, calibration, crop_area_nm2, **kw)
        for method, clustered_frac, p in (("fixed", fixed_df, fixed), ("corrected", corr_df, corr)):
            bb, diam, n = mean_sizes(clustered_frac)
            rows.append({"fraction": frac, "density_per_um2": len(df) / crop_area_nm2 * 1e6, "method": method,
                         "eps": p["eps"], "min_samples": p["min_samples"],
                         "mean_bb_size_nm": bb, "mean_diameter_nm": diam, "n_domains": n})
    size_check = pd.DataFrame(rows)
    size_check.to_csv(OUTPUT_DIR / "size_vs_density.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for ax, method, title in ((axes[0], "fixed", "Fixed eps / min_samples"),
                              (axes[1], "corrected", "Density-corrected")):
        g = size_check[size_check["method"] == method].dropna(subset=["mean_bb_size_nm"])
        r2 = linregress(g["density_per_um2"], g["mean_bb_size_nm"]).rvalue ** 2 if len(g) > 2 else np.nan
        ax.plot(g["density_per_um2"], g["mean_bb_size_nm"], "o", color=COLOR, label="bounding box")
        ax.plot(g["density_per_um2"], g["mean_diameter_nm"], "s", mfc="none", color=COLOR,
                label=f"p{SIZE_PERCENTILE:.0f} diameter")
        ax.axhline(calibration["gt_target"], color="0.4", ls=":",
                   label=f"ground truth ({calibration['gt_target']:.0f} nm)")
        ax.set_title(f"{title}\nR$^2$ (size vs density) = {r2:.3f}")
        ax.set_xlabel("Localization density (locs/um$^2$)")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel(f"{LABEL} nanodomain size (nm)")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 3. Density-corrected DBSCAN on the real cell

Density = localizations / nucleus mask area (the same definition as the calibration crop's). `eps` and `min_samples` are read off the calibration curves at that density.

In [ ]:
clustered, params = hmt.cluster.cluster_dbscan_density_corrected(
    masked, calibration, area_nm2, use_z=USE_Z, min_eps=MIN_EPS_NM, min_min_samples=MIN_MIN_SAMPLES)

lo, hi = min(calibration["densities"]), max(calibration["densities"])
extrapolated = not (lo * (1 - EXTRAPOLATION_TOLERANCE) <= params["density"] <= hi * (1 + EXTRAPOLATION_TOLERANCE))
frac_clustered = (clustered["cluster_label"] != -1).mean()

print(f"density {params['density'] * 1e6:.0f} locs/um^2 "
      f"(calibrated range {lo * 1e6:.0f}-{hi * 1e6:.0f}) -> eps {params['eps']:.1f} nm, min_samples {params['min_samples']}")
print(f"{clustered['cluster_label'].max() + 1:,} clusters | {frac_clustered:.0%} of localizations clustered")
if extrapolated:
    print("WARNING: this cell's density is outside the calibrated range, so the fits were extrapolated")
largest = clustered.loc[clustered["cluster_label"] != -1, "cluster_label"].value_counts().max() / len(clustered)
if largest > 0.05:
    print(f"WARNING: the largest cluster holds {largest:.0%} of all localizations, so DBSCAN has percolated "
          "(chained domains into one network). Check the calibration fit: min_samples may have hit "
          "MIN_MIN_SAMPLES, or eps is too large for this density.")

## 4. Nanodomain characteristics

In [ ]:
nanodomains = hmt.cluster.calculate_nanodomain_characteristics(clustered, percentile=SIZE_PERCENTILE)
kept = nanodomains[nanodomains["n_locs"] >= MIN_DOMAIN_LOCS]

print(f"{len(nanodomains):,} nanodomains ({len(kept):,} with >= {MIN_DOMAIN_LOCS} locs)")
kept[["n_locs", "bb_size_2D", "geometric_diameter_2D", "hull_size_2D", "density_2D"]].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
size_bins = np.linspace(0, np.percentile(kept["bb_size_2D"], 99), 40)
axes[0].hist(kept["bb_size_2D"], bins=size_bins, color=COLOR, edgecolor="white")
axes[0].axvline(kept["bb_size_2D"].median(), color="0.2", ls="--",
                label=f"median {kept['bb_size_2D'].median():.0f} nm")
axes[0].set_xlabel("Nanodomain size, bounding box (nm)")
axes[0].legend()

axes[1].hist(kept["geometric_diameter_2D"], bins=size_bins, color=COLOR, edgecolor="white")
axes[1].axvline(kept["geometric_diameter_2D"].median(), color="0.2", ls="--",
                label=f"median {kept['geometric_diameter_2D'].median():.0f} nm")
axes[1].set_xlabel(f"Nanodomain p{SIZE_PERCENTILE:.0f} diameter (nm)")
axes[1].legend()

axes[2].hist(kept["n_locs"], bins=np.logspace(np.log10(MIN_DOMAIN_LOCS), np.log10(kept["n_locs"].max()), 40),
             color=COLOR, edgecolor="white")
axes[2].set_xscale("log")
axes[2].set_xlabel("Localizations per nanodomain")
for ax in axes:
    ax.set_ylabel("Nanodomains")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{LABEL}: {DATA_PATH.stem}")
fig.tight_layout()
plt.show()

### Cluster map

Whole nucleus on the left; the right panel zooms into `ZOOM_NM` around the nucleus centre so individual domains are visible. Noise is gray.

In [ ]:
ZOOM_NM = 3000.0

noise = clustered["cluster_label"] == -1
colors = np.array(clustered["cluster_color"].tolist())
cx, cy = masked["x [nm]"].median(), masked["y [nm]"].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, s in ((axes[0], 0.05), (axes[1], 2)):
    ax.scatter(clustered.loc[noise, "x [nm]"], clustered.loc[noise, "y [nm]"], s=s, color="0.8", linewidths=0)
    ax.scatter(clustered.loc[~noise, "x [nm]"], clustered.loc[~noise, "y [nm]"], s=s, c=colors[~noise.to_numpy()],
               linewidths=0)
    ax.set_aspect("equal")
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("y (nm)")
axes[0].add_patch(plt.Rectangle((cx - ZOOM_NM / 2, cy - ZOOM_NM / 2), ZOOM_NM, ZOOM_NM, fill=False, color="k"))
axes[0].set_title(f"{LABEL}: eps {params['eps']:.0f} nm, min_samples {params['min_samples']}")
axes[1].set_xlim(cx - ZOOM_NM / 2, cx + ZOOM_NM / 2)
axes[1].set_ylim(cy - ZOOM_NM / 2, cy + ZOOM_NM / 2)
axes[1].set_title(f"{ZOOM_NM / 1000:.0f} um zoom")
fig.tight_layout()
plt.show()

Optional: interactive 3D view in napari. `plot_clusters_napari` takes two channels, so the second is empty.

In [ ]:
# hmt.visualize.plot_clusters_napari(clustered, clustered.iloc[:0])

## 5. Save

In [ ]:
keep_cols = [c for c in ["id", "frame", "x [nm]", "y [nm]", "z [nm]", "uncertainty [nm]", "cluster_label"]
             if c in clustered.columns and (HAS_Z or c != "z [nm]")]
clustered[keep_cols].to_csv(OUTPUT_DIR / "clustered_locs.csv", index=False)
drop_cols = ["cluster_color", "centroid", "vertices"]
if not HAS_Z:  # 3D measures are meaningless on the z = 0 placeholder
    drop_cols += ["centroid_z", "hull_size_3D", "bb_size_3D", "density_3D"]
nanodomains.drop(columns=drop_cols).to_csv(OUTPUT_DIR / "nanodomains.csv", index=False)

summary = {
    "data_path": str(DATA_PATH), "label": LABEL,
    "mask_thresh": thresh, "mask_area_um2": area_nm2 / 1e6,
    "n_locs_raw": len(raw), "n_locs_masked": len(masked),
    "density_per_um2": params["density"] * 1e6,
    "eps_nm": params["eps"], "min_samples": params["min_samples"], "has_z": HAS_Z, "use_z": USE_Z,
    "calibration": str(CALIBRATION_JSON or OUTPUT_DIR / "calibration.json"),
    "calibrated_density_range_per_um2": [lo * 1e6, hi * 1e6], "extrapolated": extrapolated,
    "frac_locs_clustered": frac_clustered,
    "n_nanodomains": len(nanodomains), f"n_nanodomains_min{MIN_DOMAIN_LOCS}locs": len(kept),
    "mean_bb_size_nm": kept["bb_size_2D"].mean(), "median_bb_size_nm": kept["bb_size_2D"].median(),
    "mean_diameter_nm": kept["geometric_diameter_2D"].mean(),
}
with open(OUTPUT_DIR / "summary.json", "w") as f:
    json.dump(to_jsonable(summary), f, indent=2)
print(f"outputs -> {OUTPUT_DIR}")
to_jsonable(summary)